# Bài tập 1: Huấn luyện mô hình Logistic Regression với bài toán AND

## 1. Tính đạo hàm theo giải thuật Back-Propagation (đã làm trên lớp)

### Forward Pass
Với một mẫu dữ liệu $(x_1, x_2, y)$:

**Bước 1 — Linear**
$$z = w_1 x_1 + w_2 x_2 + b$$

**Bước 2 — Sigmoid (Activation)**
$$a = \sigma(z) = \frac{1}{1 + e^{-z}}$$

**Bước 3 — Loss (Binary Cross-Entropy)**
$$L(a,y) = -\big[y \ln(a) + (1-y)\ln(1-a)\big]$$


### Mục tiêu của Back-Propagation
Cần tính gradient của hàm mất mát theo từng tham số:
$$\frac{\partial L}{\partial w_1}, \quad \frac{\partial L}{\partial w_2}, \quad \frac{\partial L}{\partial b}$$

Để cập nhật bằng Gradient Descent:
$$w := w - \alpha \frac{\partial L}{\partial w}$$


### Chuỗi phụ thuộc (Computation Graph)
$$w_1, w_2, b \rightarrow z \rightarrow a \rightarrow L$$

Áp dụng quy tắc dây chuyền (Chain Rule):
$$\frac{\partial L}{\partial w} = \frac{\partial L}{\partial a} \cdot \frac{\partial a}{\partial z} \cdot \frac{\partial z}{\partial w}$$

---

## Các đạo hàm thành phần

### 1. Đạo hàm của Loss theo $a$
$$L = -\big[y\ln a + (1-y)\ln(1-a)\big]$$
$$\frac{\partial L}{\partial a} = -\left(\frac{y}{a} - \frac{1-y}{1-a}\right) = \frac{a-y}{a(1-a)}$$

### 2. Đạo hàm của Sigmoid theo $z$
$$a = \sigma(z) \Rightarrow \frac{\partial a}{\partial z} = a(1-a)$$

### 3. Đạo hàm của Loss theo $z$ (Lỗi dự báo)
Áp dụng Chain Rule:
$$\frac{\partial L}{\partial z} = \frac{\partial L}{\partial a} \cdot \frac{\partial a}{\partial z}$$
$$\frac{\partial L}{\partial z} = \left(\frac{a-y}{a(1-a)}\right) \cdot a(1-a) = a - y$$
Đặt: $$dz \equiv \frac{\partial L}{\partial z} = a - y$$

### 4. Đạo hàm của $z$ theo các tham số
$$z = w_1x_1 + w_2x_2 + b$$
$$\frac{\partial z}{\partial w_1} = x_1, \quad \frac{\partial z}{\partial w_2} = x_2, \quad \frac{\partial z}{\partial b} = 1$$

---

## Kết quả Gradient cuối cùng
Áp dụng Chain Rule kết hợp các bước trên:

$$\frac{\partial L}{\partial w_1} = \frac{\partial L}{\partial z} \cdot \frac{\partial z}{\partial w_1} = (a-y)x_1$$
$$\frac{\partial L}{\partial w_2} = \frac{\partial L}{\partial z} \cdot \frac{\partial z}{\partial w_2} = (a-y)x_2$$
$$\frac{\partial L}{\partial b} = \frac{\partial L}{\partial z} \cdot \frac{\partial z}{\partial b} = (a-y)$$

**Tóm tắt hệ thức cập nhật:**
$$\boxed{
\begin{aligned}
\frac{\partial L}{\partial w_1} &= x_1(a-y) \\
\frac{\partial L}{\partial w_2} &= x_2(a-y) \\
\frac{\partial L}{\partial b} &= (a-y)
\end{aligned}
}$$

## 2. Thuật toán huấn luyện mô hình phân lớp nhị phân cho Dataset gồm M mẫu

In [84]:
import numpy as np

In [85]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def forward(w1, w2, x1, x2, b):
    z = w1 * x1 + w2 * x2 + b
    a = sigmoid(z)
    return a

def compute_loss(a, y):
    eps = 1e-9
    return - (y * np.log(a + eps) + (1 - y) * np.log(1 - a + eps))

In [86]:
def train_model(X, Y, learning_rate=0.01, epochs=200):
    number_samples, number_features = X.shape

    w1 = np.random.rand()
    w2 = np.random.rand()
    b = np.random.rand()
    logs = []

    for epoch in range(epochs):
        sum_dw1, sum_dw2, sum_db, sum_loss = 0, 0, 0, 0
        for i in range(number_samples):
            x1, x2, y = X[i, 0], X[i, 1], Y[i]
            a = forward(w1, w2, x1, x2, b)
            loss = compute_loss(a, y)

            dz = a - y
            sum_dw1 += x1 * dz
            sum_dw2 += x2 * dz
            sum_db += dz
            sum_loss += loss

        w1 -= learning_rate * sum_dw1 / number_samples
        w2 -= learning_rate * sum_dw2 / number_samples
        b -= learning_rate * sum_db / number_samples
        logs.append(sum_loss / number_samples)

        # if epoch % 10 == 0:
        #     print(f"Epoch {epoch}: loss = {logs[-1]}")

    return w1, w2, b, logs

In [87]:
# Test
M = 150
X = np.random.rand(M, 2)
Y = np.random.randint(0, 2, size=(M, 1))

print("3 sample of X: ", X[:3])
print("3 sample of Y: ", Y[:3])

w1, w2, b, logs = train_model(X, Y, learning_rate=0.01, epochs=200)
print("w1: ", w1)
print("w2: ", w2)
print("b: ", b)
print("logs: ", logs)

3 sample of X:  [[0.78786407 0.62078565]
 [0.25131685 0.38228421]
 [0.67493491 0.08613737]]
3 sample of Y:  [[1]
 [1]
 [1]]
w1:  [0.54900832]
w2:  [0.59103636]
b:  [-0.01267406]
logs:  [array([0.90106121]), array([0.89983263]), array([0.89861018]), array([0.89739385]), array([0.89618363]), array([0.89497949]), array([0.89378142]), array([0.8925894]), array([0.89140341]), array([0.89022344]), array([0.88904947]), array([0.88788148]), array([0.88671945]), array([0.88556337]), array([0.88441321]), array([0.88326897]), array([0.88213061]), array([0.88099813]), array([0.87987151]), array([0.87875072]), array([0.87763575]), array([0.87652658]), array([0.87542319]), array([0.87432556]), array([0.87323368]), array([0.87214752]), array([0.87106707]), array([0.86999231]), array([0.86892322]), array([0.86785978]), array([0.86680197]), array([0.86574977]), array([0.86470317]), array([0.86366214]), array([0.86262667]), array([0.86159673]), array([0.86057231]), array([0.85955338]), array([0.85853993

## 3. Chuyển thuật toán sang dạng véc tơ hóa

In [88]:
def train_vectorized(X, Y, learning_rate=0.01, epochs=200):
    M, N = X.shape
    X_new = np.concatenate([np.ones(M).reshape(-1, 1), X], axis=1)
    N_new = X_new.shape[1]

    W_new = np.random.rand(N_new, 1) * 0.01
    logs = []

    for epoch in range(epochs):
        Z = np.dot(X_new, W_new)
        A = 1.0 / (1.0 + np.exp(-Z))
        eps = 1e-9

        loss = np.mean( - (Y * np.log(A + eps) + (1 - Y) * np.log(1 - A + eps)))
        logs.append(loss)

        dZ = A - Y
        d_W_new = np.dot(X_new.T, dZ)
        W_new -= (1.0 / M) * learning_rate * d_W_new

        # if epoch % 10 == 0:
        #     print(f"Epoch {epoch}: loss = {logs[-1]}")

    return W_new, logs

In [89]:
w, logs = train_vectorized(X, Y, learning_rate=0.01, epochs=200)
print("w: ", w)
print("logs: ", logs)

w:  [[-0.03436469]
 [ 0.01038757]
 [-0.05520221]]
logs:  [np.float64(0.6934752353166644), np.float64(0.6934545252002239), np.float64(0.6934339307677387), np.float64(0.6934134511930525), np.float64(0.693393085655973), np.float64(0.6933728333422304), np.float64(0.693352693443434), np.float64(0.6933326651570303), np.float64(0.6933127476862617), np.float64(0.6932929402401247), np.float64(0.6932732420333283), np.float64(0.6932536522862537), np.float64(0.6932341702249132), np.float64(0.6932147950809102), np.float64(0.6931955260913987), np.float64(0.6931763624990437), np.float64(0.6931573035519819), np.float64(0.6931383485037825), np.float64(0.6931194966134074), np.float64(0.6931007471451739), np.float64(0.6930820993687146), np.float64(0.6930635525589411), np.float64(0.6930451059960044), np.float64(0.693026758965259), np.float64(0.6930085107572234), np.float64(0.6929903606675453), np.float64(0.6929723079969633), np.float64(0.6929543520512708), np.float64(0.6929364921412794), np.float64(0.6929

In [90]:
# Compare time
import time

start_1 = time.time()
_, _, _, _ = train_model(X, Y, learning_rate=0.01, epochs=2000)
end_1 = time.time()
total_time_1 = end_1 - start_1

start_2 = time.time()
_, _ = train_vectorized(X, Y, learning_rate=0.01, epochs=2000)
end_2 = time.time()
total_time_2 = end_2 - start_2

print(">" * 20, "Time comparison", "<" * 20)
print(f"Time for train_model: {total_time_1:.2f}s")
print(f"Time for train_vectorized: {total_time_2:.2f}s")

>>>>>>>>>>>>>>>>>>>> Time comparison <<<<<<<<<<<<<<<<<<<<
Time for train_model: 3.53s
Time for train_vectorized: 0.03s


## 4. Thử nghiệm cho bài toán AND

In [98]:
X_and = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
])
Y_and = np.array([
    [0],
    [0],
    [0],
    [1]
]).reshape(-1, 1)

w_and, logs_and = train_vectorized(X_and, Y_and, learning_rate=0.01, epochs=2000)
print("w: ", w_and)

# Test
X_and_test = np.concatenate([np.ones(4).reshape(-1, 1), X_and], axis=1)
Z_and_test = np.dot(X_and_test, w_and)
A_and_test = 1.0 / (1.0 + np.exp(-Z_and_test))
predictions = (A_and_test >= 0.5).astype(int)

for i in range (4):
    print(f"Input {X_and[i, :]}, Output {Y_and[i][0]}, Prediction {predictions[i][0]} with {A_and_test[i][0] * 100: .2f} %")

w:  [[-1.91872521]
 [ 1.02098519]
 [ 1.0211943 ]]
Input [0 0], Output 0, Prediction 0 with  12.80 %
Input [0 1], Output 0, Prediction 0 with  28.96 %
Input [1 0], Output 0, Prediction 0 with  28.95 %
Input [1 1], Output 1, Prediction 1 with  53.08 %
